# EVALUASI PRECISION@K — Notebook Terpadu

Notebook ini menggabungkan seluruh tahapan evaluasi dalam satu alur:

1. **Fix Ground Truth IDs** — memetakan ulang ID artikel lama ke ID baru berdasarkan pencocokan judul
2. **Load TF-IDF & Metadata** — memuat matriks TF-IDF, vektor, dan metadata artikel dari Supabase
3. **Search Semua Query** — menjalankan pencarian untuk 12 query uji bilingual
4. **Gabung dengan Ground Truth** — mencocokkan hasil pencarian dengan label final dari 3 evaluator
5. **Hitung Precision@K** — menghitung P@5, P@10, dan P@20
6. **Simpan Hasil** — menyimpan ke CSV dan Supabase
7. **Cetak Ringkasan** — menampilkan tabel hasil


## 1. Installasi Dependensi

In [ ]:
%pip install supabase python-dotenv pandas numpy scipy scikit-learn PySastrawi

## 2. Import Library dan Koneksi Supabase

In [ ]:
# =========================================================
# CELL 2 - IMPORT LIBRARY + KONEKSI SUPABASE
# =========================================================
import os
import re
import json
import sys
import csv
import scipy.sparse as sp
import pandas as pd
import numpy as np

from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from dotenv import load_dotenv
from supabase import create_client
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.metrics.pairwise import cosine_similarity

BACKEND_DIR = next(
    path for path in (Path.cwd(), Path.cwd().parent, Path.cwd() / "backend")
    if (path / "src" / "preprocessing").is_dir()
)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from src.preprocessing.stopwords import get_stopwords

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY") or os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

SOURCE_TABLE = "cleaned_papers_results"
EVAL_TABLE = "evaluation_precision_at_k"

base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
tfidf_dir = os.path.join(base_dir, "data", "tfidf")
eval_dir = os.path.join(base_dir, "data", "evaluation")

os.makedirs(eval_dir, exist_ok=True)

print("✅ Import dan koneksi Supabase berhasil")

## 3. Load File VSM (TF-IDF Matrix)

In [ ]:
# =========================================================
# CELL 3 - LOAD FILE VSM HASIL TF-IDF
# =========================================================
print("[1] Loading TF-IDF...")
tfidf_matrix = sp.load_npz(os.path.join(tfidf_dir, "tfidf_matrix.npz"))

with open(os.path.join(tfidf_dir, "tfidf_terms.json"), "r", encoding="utf-8") as f:
    terms = json.load(f)

with open(os.path.join(tfidf_dir, "tfidf_doc_ids.json"), "r", encoding="utf-8") as f:
    doc_ids = json.load(f)

with open(os.path.join(tfidf_dir, "idf_scores.json"), "r", encoding="utf-8") as f:
    idf_scores = json.load(f)

tfidf_documents = pd.read_csv(os.path.join(tfidf_dir, "tfidf_documents.csv"))

doc_ids = [int(doc_id) for doc_id in doc_ids]

print("✅ File VSM berhasil dimuat")
print(f"Matrix TF-IDF: {tfidf_matrix.shape}")
print(f"Jumlah terms: {len(terms)}")
print(f"Jumlah doc_ids: {len(doc_ids)}")

## 4. Load Metadata Artikel dari Supabase

In [ ]:
# =========================================================
# CELL 4 - LOAD METADATA ARTIKEL DARI SUPABASE
# =========================================================
print("[2] Loading metadata from Supabase...")
all_data = []
batch_size = 1000
offset = 0

selected_columns = "id,title,abstract,authors,year,source,category,pdf_url,url,scrape_status"

while True:
    response = (
        supabase.table(SOURCE_TABLE)
        .select(selected_columns)
        .range(offset, offset + batch_size - 1)
        .execute()
    )
    batch = response.data or []
    if not batch:
        break
    all_data.extend(batch)
    if len(batch) < batch_size:
        break
    offset += batch_size

metadata_df = pd.DataFrame(all_data)

if metadata_df.empty:
    raise ValueError("❌ Data cleaned_papers_results kosong.")

metadata_df["id"] = metadata_df["id"].astype("int64")

doc_index = pd.DataFrame({"id": doc_ids})
doc_index = doc_index.merge(metadata_df, on="id", how="left")
doc_index = doc_index.merge(
    tfidf_documents[["id", "document_text"]],
    on="id",
    how="left"
)

for col in ["title", "abstract", "authors", "source", "category", "pdf_url", "url", "scrape_status", "document_text"]:
    if col in doc_index.columns:
        doc_index[col] = doc_index[col].fillna("")

if tfidf_matrix.shape[0] != len(doc_index):
    raise ValueError("❌ Jumlah matrix TF-IDF tidak sama dengan jumlah dokumen.")

print("✅ Metadata artikel siap")
print(f"Jumlah dokumen: {len(doc_index)}")
doc_index.head(3)

## 5. Load Stopwords dan Stemmer

In [ ]:
# =========================================================
# CELL 5 - STOPWORDS + STEMMER
# =========================================================
print("[3] Loading stopwords + stemmer...")
stop_words = get_stopwords()
stemmer = StemmerFactory().create_stemmer()
print("✅ Stopwords dan stemmer siap")

## 6. Definisi Fungsi Pencarian (Query → Vektor → Cosine Similarity)

Fungsi-fungsi ini merupakan inti dari mesin pencarian:
- `preprocess_query()`: membersihkan, tokenisasi, stopword removal, stemming
- `build_query_vector()`: membentuk vektor TF-IDF query
- `count_occurrence()`: menghitung frekuensi term query dalam dokumen
- `interpret_similarity()`: menginterpretasi skor cosine similarity
- `search()`: pipeline pencarian lengkap


In [ ]:
# =========================================================
# CELL 6 - FUNGSI QUERY, COSINE, DAN SEARCH
# =========================================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_query(query):
    cleaned = clean_text(query)
    tokens = cleaned.split()
    tokens = [token for token in tokens if token not in stop_words and len(token) > 1]
    tokens = [stemmer.stem(token) for token in tokens]
    return tokens

def build_query_vector(query_tokens):
    term_to_index = {term: index for index, term in enumerate(terms)}
    query_vector = np.zeros(len(terms), dtype=float)
    if not query_tokens:
        return query_vector.reshape(1, -1)
    total_terms = len(query_tokens)
    term_counts = Counter(query_tokens)
    for term, count in term_counts.items():
        if term in term_to_index:
            tf = count / total_terms
            idf = float(idf_scores.get(term, 0))
            query_vector[term_to_index[term]] = tf * idf
    return query_vector.reshape(1, -1)

def count_occurrence(document_text, query_tokens):
    document_tokens = str(document_text).split()
    return sum(document_tokens.count(term) for term in query_tokens)

def interpret_similarity(score, threshold=0.3):
    if score > 0.5:
        return "Relevan Tinggi"
    if score > threshold:
        return "Relevan Sedang"
    return "Rendah"

def search(query, top_k=20, min_occurrence=0, threshold=0.3):
    query_tokens = preprocess_query(query)
    if not query_tokens:
        return pd.DataFrame()
    query_vector = build_query_vector(query_tokens)
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    results = doc_index.copy()
    results["similarity_score"] = scores
    results["occurrence"] = results["document_text"].apply(
        lambda text: count_occurrence(text, query_tokens)
    )
    results = results[results["similarity_score"] > 0].copy()
    if min_occurrence > 0:
        results = results[results["occurrence"] >= min_occurrence].copy()
    results = results.sort_values("similarity_score", ascending=False)
    results = results.head(top_k).reset_index(drop=True)
    results["rank"] = range(1, len(results) + 1)
    results["threshold_relevan"] = (results["similarity_score"] > threshold).astype(int)
    results["interpretation"] = results["similarity_score"].apply(
        lambda score: interpret_similarity(score, threshold)
    )
    return results

print("✅ Fungsi search evaluasi siap")

## 7. Query Uji Evaluasi (Bilingual)

12 query uji yang mewakili 4 kategori dengan variasi Bahasa Indonesia dan Inggris.
Nilai K = 5, 10, 20 sesuai proposal.


In [ ]:
# =========================================================
# CELL 7 - QUERY UJI EVALUASI
# K = 5, 10, 20 sesuai proposal
# Query bilingual karena dataset berisi artikel Indonesia dan Inggris.
# =========================================================
queries_eval = [
    {"query": "machine learning", "kategori": "Machine Learning"},
    {"query": "pembelajaran mesin", "kategori": "Machine Learning"},
    {"query": "data mining", "kategori": "Machine Learning"},

    {"query": "web application", "kategori": "Web Application"},
    {"query": "aplikasi web", "kategori": "Web Application"},
    {"query": "sistem informasi berbasis web", "kategori": "Web Application"},

    {"query": "cyber security", "kategori": "Cyber Security"},
    {"query": "keamanan siber", "kategori": "Cyber Security"},
    {"query": "keamanan jaringan", "kategori": "Cyber Security"},

    {"query": "mobile application", "kategori": "Mobile Application"},
    {"query": "aplikasi mobile", "kategori": "Mobile Application"},
    {"query": "aplikasi android", "kategori": "Mobile Application"},
]

K_VALUES = [5, 10, 20]
MAX_K = max(K_VALUES)
THRESHOLD = 0.3

print(f"✅ {len(queries_eval)} query evaluasi bilingual siap")
print(f"K_VALUES = {K_VALUES}, MAX_K = {MAX_K}, THRESHOLD = {THRESHOLD}")

## 8. Jalankan Search untuk Semua Query

In [ ]:
# =========================================================
# CELL 8 - JALANKAN SEARCH UNTUK SEMUA QUERY
# =========================================================
print("[5] Running search for all queries...")
all_results = {}
kemunculan_all = defaultdict(int)
kemunculan_kat = defaultdict(lambda: defaultdict(int))

for item in queries_eval:
    query = item["query"]
    kategori = item["kategori"]
    results = search(query=query, top_k=MAX_K, min_occurrence=0, threshold=THRESHOLD)
    all_results[query] = results
    print(f"  {query}: {len(results)} hasil")

    for _, row in results.iterrows():
        kemunculan_all[row["title"]] += 1
        kemunculan_kat[kategori][row["title"]] += 1

print("✅ Semua query selesai diproses")

## 9. Fix Ground Truth IDs

Memetakan ulang ID artikel lama → ID baru dengan mencocokkan judul artikel 
antara ground truth CSV dan hasil pencarian saat ini.

*Fungsi ini menggantikan `fix_ground_truth_ids.py`*.


In [ ]:
# =========================================================
# CELL 9 - FIX GROUND TRUTH IDs
# Menggantikan fix_ground_truth_ids.py
# =========================================================
print("[6] Loading ground truth CSV & Fixing IDs...")

def normalize_title(title):
    return re.sub(r'\s+', ' ', str(title).lower().strip())

gt_raw_path = os.path.join(eval_dir, "ground_truth_raw_from_sheets.csv")

if not os.path.exists(gt_raw_path):
    gt_path = os.path.join(eval_dir, "ground_truth_evaluation_3_evaluator.csv")
    print(f"  File raw tidak ditemukan, langsung pakai: {gt_path}")
else:
    gt_df = pd.read_csv(gt_raw_path)
    print(f"  Raw entries: {len(gt_df)}")

    title_mapping = {}
    unmatched_entries = []
    total_matched = 0

    for item in queries_eval:
        query = item["query"]
        current_results = all_results[query]
        current_by_title = {}
        for _, row in current_results.iterrows():
            norm = normalize_title(row["title"])
            current_by_title[norm] = int(row["id"])

        query_gt = gt_df[gt_df["Query"].str.lower().str.strip() == query.lower()]
        for _, gt_row in query_gt.iterrows():
            old_id = int(gt_row["Article ID"])
            gt_title_norm = normalize_title(gt_row.get("Judul Artikel", ""))
            if gt_title_norm in current_by_title:
                title_mapping[old_id] = current_by_title[gt_title_norm]
                total_matched += 1
            else:
                unmatched_entries.append({
                    "query": query, "old_id": old_id, "title": gt_row.get("Judul Artikel", "")
                })

    print(f"  Matched: {total_matched}/{len(gt_df)}")
    print(f"  Unmatched: {len(unmatched_entries)}")

    if unmatched_entries:
        print("  Contoh unmatched (first 5):")
        for e in unmatched_entries[:5]:
            print(f"    Query='{e['query']}' Old ID={e['old_id']} Title='{e['title'][:60]}...'")

    gt_df["Article ID (Old)"] = gt_df["Article ID"].copy()
    gt_df["Article ID"] = gt_df["Article ID"].apply(lambda x: title_mapping.get(int(x), int(x)))
    gt_df["ID Match"] = gt_df.apply(
        lambda r: "Matched" if int(r["Article ID (Old)"]) in title_mapping else "Unmatched", axis=1
    )

    gt_df_filtered = gt_df[gt_df["ID Match"] == "Matched"].copy()
    print(f"  After filtering unmatched: {len(gt_df_filtered)} entries remaining")

    gt_df.to_csv(os.path.join(eval_dir, "ground_truth_mapped_debug.csv"), index=False)

    save_cols = [c for c in gt_df_filtered.columns if c not in ["Article ID (Old)", "ID Match"]]
    gt_df_filtered[save_cols].to_csv(
        os.path.join(eval_dir, "ground_truth_evaluation_3_evaluator.csv"), index=False
    )
    print("  Saved: ground_truth_evaluation_3_evaluator.csv")

# Load the GT file (whether just created or already existed)
gt_path = os.path.join(eval_dir, "ground_truth_evaluation_3_evaluator.csv")
evaluator_df = pd.read_csv(gt_path)
print(f"  Final ground truth entries: {len(evaluator_df)}")
print("✅ Ground truth siap")

## 10. Susun Hasil Pencarian dan Gabung dengan Ground Truth

In [ ]:
# =========================================================
# CELL 10 - SUSUN HASIL PENCARIAN + MERGE DENGAN GROUND TRUTH
# =========================================================
print("[7] Building search results dataframe...")
rows = []
for item in queries_eval:
    query = item["query"]
    kategori_query = item["kategori"]
    results = all_results[query]
    for _, row in results.iterrows():
        rows.append({
            "query": query,
            "kategori_query": kategori_query,
            "rank": int(row["rank"]),
            "article_id": int(row["id"]),
            "article_category": row["category"],
            "title": row["title"],
            "abstract": row["abstract"],
            "similarity_score": float(row["similarity_score"]),
            "threshold_relevan": int(row["threshold_relevan"]),
        })

hasil_pencarian_eval_df = pd.DataFrame(rows)
print(f"  Total search results: {len(hasil_pencarian_eval_df)}")

print("[8] Merging with ground truth...")
evaluator_df["query_key"] = evaluator_df["Query"].astype(str).str.lower().str.strip()
evaluator_df["article_id"] = pd.to_numeric(evaluator_df["Article ID"], errors="coerce").astype("Int64")

gt_df = hasil_pencarian_eval_df.copy()
gt_df["query_key"] = gt_df["query"].astype(str).str.lower().str.strip()
gt_df["article_id"] = pd.to_numeric(gt_df["article_id"], errors="coerce").astype("Int64")

eval_cols = ["query_key", "article_id", "Label Final", "Status Final",
             "Evaluator 1", "Evaluator 2", "Evaluator 3", "Catatan Evaluator",
             "Threshold Awal", "Similarity Score"]
merge_cols = [c for c in eval_cols if c in evaluator_df.columns]
gt_df = gt_df.merge(evaluator_df[merge_cols], on=["query_key", "article_id"], how="left")

gt_df["relevan"] = pd.to_numeric(gt_df["Label Final"], errors="coerce").fillna(0).astype(int)
gt_df["query"] = gt_df["query"].astype(str).str.lower().str.strip()
gt_df["article_id"] = gt_df["article_id"].astype("int64")
gt_df["relevan"] = gt_df["relevan"].astype(int)

print(f"  Total after merge: {len(gt_df)}")
print(f"  Distribusi relevan:\n{gt_df['relevan'].value_counts().sort_index()}")
print("✅ Data siap untuk perhitungan Precision@K")

## 11. Perhitungan Precision@K

Untuk setiap query, hitung Precision@5, @10, dan @20 dengan rumus:

$$P@K = \frac{\text{jumlah artikel relevan di peringkat 1..K}}{K}$$

In [ ]:
# =========================================================
# CELL 11 - HITUNG PRECISION@K
# =========================================================
print("[9] Calculating Precision@K...")

# --- TABEL 1: Hasil Pencarian Detail ---
tabel1_rows = []
for item in queries_eval:
    query = item["query"]
    kategori_query = item["kategori"]
    results = all_results[query].copy()
    sub_gt = gt_df[gt_df["query"] == query][["article_id", "relevan"]].copy()
    sub_gt = sub_gt.rename(columns={"article_id": "id"})
    merged = results.merge(sub_gt, on="id", how="left")
    merged["relevan"] = merged["relevan"].fillna(0).astype(int)
    for _, row in merged.iterrows():
        tabel1_rows.append({
            "Query": query,
            "Kategori Query": kategori_query,
            "Rank": int(row["rank"]),
            "Article ID": int(row["id"]),
            "Kategori Artikel": row["category"],
            "Judul": row["title"],
            "Penulis": row["authors"],
            "Tahun": row["year"],
            "Source": row["source"],
            "Similarity Score": round(float(row["similarity_score"]), 6),
            "Threshold Relevan": "Ya" if row["threshold_relevan"] == 1 else "Tidak",
            "Relevan Human Judgment": "Ya" if row["relevan"] == 1 else "Tidak",
            "Occurrence": int(row["occurrence"]),
            "Interpretasi": row["interpretation"]
        })

tabel1_df = pd.DataFrame(tabel1_rows)
tabel1_path = os.path.join(eval_dir, "tabel1_hasil_pencarian.csv")
tabel1_df.to_csv(tabel1_path, index=False)
print(f"  Tabel 1 saved: {tabel1_path} ({len(tabel1_df)} rows)")

# --- TABEL 2: Precision@K per Query ---
eval_rows = []
for item in queries_eval:
    query = item["query"]
    kategori_query = item["kategori"]
    results = all_results[query].copy()
    sub_gt = gt_df[gt_df["query"] == query][["article_id", "relevan"]].copy()
    sub_gt = sub_gt.rename(columns={"article_id": "id"})
    merged = results.merge(sub_gt, on="id", how="left")
    merged["relevan"] = merged["relevan"].fillna(0).astype(int)

    row_eval = {"Query": query, "Kategori": kategori_query, "Retrieved": int(len(merged))}
    for k in K_VALUES:
        top_k = merged.head(k)
        relevant_k = int(top_k["relevan"].sum())
        precision_k = relevant_k / k
        row_eval[f"Relevan@{k}"] = relevant_k
        row_eval[f"P@{k}"] = round(precision_k, 4)
    eval_rows.append(row_eval)

eval_df = pd.DataFrame(eval_rows)
eval_path = os.path.join(eval_dir, "tabel2_precision_at_k.csv")
eval_df.to_csv(eval_path, index=False)
print(f"  Tabel 2 saved: {eval_path}")

print("\n  Precision@K per query:")
print(eval_df.to_string(index=False))

## 12. Rata-rata Precision@K

Hitung rata-rata P@K untuk semua query (keseluruhan dan per kategori).


In [ ]:
# =========================================================
# CELL 12 - RATA-RATA PRECISION@K
# =========================================================
print("[10] Calculating averages...")

# --- Rata-rata Keseluruhan ---
summary_rows = []
for k in K_VALUES:
    mean_val = round(eval_df[f"P@{k}"].mean(), 4)
    summary_rows.append({
        "Metrik": f"Mean P@{k}",
        "Nilai": mean_val,
        "Persentase": round(mean_val * 100, 2)
    })
summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(eval_dir, "rata_rata_keseluruhan.csv")
summary_df.to_csv(summary_path, index=False)

print("\n  Rata-rata keseluruhan:")
print(summary_df.to_string(index=False))

# --- Rata-rata per Kategori ---
kat_avg_rows = []
for kategori, group in eval_df.groupby("Kategori"):
    row = {"Kategori": kategori}
    for k in K_VALUES:
        row[f"Rata-rata P@{k}"] = round(group[f"P@{k}"].mean(), 4)
    kat_avg_rows.append(row)

kat_avg_df = pd.DataFrame(kat_avg_rows)
kat_avg_path = os.path.join(eval_dir, "rata_rata_per_kategori.csv")
kat_avg_df.to_csv(kat_avg_path, index=False)

print("\n  Rata-rata per kategori:")
print(kat_avg_df.to_string(index=False))

## 13. Simpan Hasil Evaluasi ke Supabase

Menyimpan nilai Precision@K ke tabel `evaluation_precision_at_k` di Supabase.


In [ ]:
# =========================================================
# CELL 13 - SIMPAN KE SUPABASE
# =========================================================
print("[11] Saving to Supabase...")
ts = datetime.now(timezone.utc).isoformat()
rows_db = []
for _, row in eval_df.iterrows():
    query = row["Query"]
    for k in K_VALUES:
        rows_db.append({
            "compared_text": query,
            "k": int(k),
            "retrieved_count": int(row["Retrieved"]),
            "relevant_retrieved": int(row[f"Relevan@{k}"]),
            "precision_at_k": float(row[f"P@{k}"]),
            "updated_at": ts
        })

for start in range(0, len(rows_db), 500):
    batch = rows_db[start:start + 500]
    supabase.table(EVAL_TABLE).upsert(batch, on_conflict="compared_text,k").execute()

print(f"  Saved {len(rows_db)} rows to {EVAL_TABLE}")
print("\n✅ Evaluation complete!")

## 14. Ringkasan Hasil

Menampilkan contoh hasil query "cyber security" Top-5 dan seluruh tabel Precision@K.


In [ ]:
# =========================================================
# CELL 14 - CETAK RINGKASAN HASIL
# Menggantikan print_results.py
# =========================================================
print("=" * 70)
print("CONTOH: Hasil Pencarian Query 'cyber security' pada Top-5")
print("=" * 70)

cyber_rows = tabel1_df[tabel1_df["Query"] == "cyber security"].head(5)
for _, row in cyber_rows.iterrows():
    print(f"Rank {int(row['Rank'])} | ID={int(row['Article ID'])}")
    print(f"  Judul: {str(row['Judul'])[:80]}...")
    print(f"  Similarity: {row['Similarity Score']} | Threshold: {row['Threshold Relevan']} | Ground Truth: {row['Relevan Human Judgment']}")
    print()

print("=" * 70)
print("TABEL 2: Precision@K per Query")
print("=" * 70)
print(eval_df.to_string(index=False))

print("\n" + "=" * 70)
print("RATA-RATA PRECISION@K")
print("=" * 70)
print(summary_df.to_string(index=False))

print("\n" + "=" * 70)
print("RATA-RATA PER KATEGORI")
print("=" * 70)
print(kat_avg_df.to_string(index=False))

print("\n✅ Semua hasil siap untuk laporan.")